# Step 3C — Merge reference labels, corrected-expression UCell scores, and rescue rare T cells

Run this notebook back in the **ResolVI / pyUCell environment**.

For each sample it combines four independent evidence streams:

1. raw-count T-cell coherence;
2. ResolVI-corrected pyUCell scores;
3. Pan-Human hierarchical predictions and confidence;
4. CellTypist `Immune_All_High` and `Immune_All_Low` predictions.

The corrected expression is read in cell chunks from the all-cell Zarr written
by Step 3A. The scored H5AD keeps raw sparse counts in `X` and stores only
scores, reference labels, and candidate flags in metadata. The authoritative
corrected matrix remains in the Zarr layer.

No final CD4/CD8/Treg call is forced here. This notebook creates a sensitive
T/NK candidate union and a stricter multi-evidence subset for the next
integration and subclustering step.


In [1]:
# ---------------------------------------------------------------------
# Environment — run before importing torch / pyUCell
# ---------------------------------------------------------------------
import os

GPU_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["PYTHONHASHSEED"] = "0"
os.environ["OMP_NUM_THREADS"] = "16"
os.environ["MKL_NUM_THREADS"] = "16"
os.environ["OPENBLAS_NUM_THREADS"] = "16"

print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])


CUDA_VISIBLE_DEVICES: 0


In [2]:
from __future__ import annotations

import gc
import hashlib
import json
import time
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyucell as uc
import scipy.sparse as sp
import torch
import zarr

SAMPLE_INFO = {
    "Screen_39_21": {"patient": "patient_39_21", "cancer_type": "NSCLC", "biopsy_stage": "Screen"},
    "C2D15_39_21": {"patient": "patient_39_21", "cancer_type": "NSCLC", "biopsy_stage": "C2D15"},
    "Screen_17_26": {"patient": "patient_17_26", "cancer_type": "NSCLC", "biopsy_stage": "Screen"},
    "C2D15_17_26": {"patient": "patient_17_26", "cancer_type": "NSCLC", "biopsy_stage": "C2D15"},
    "Screen_18_23": {"patient": "patient_18_23", "cancer_type": "melanoma", "biopsy_stage": "Screen"},
    "C2D15_18_23": {"patient": "patient_18_23", "cancer_type": "melanoma", "biopsy_stage": "C2D15"},
    "Screen_16_22": {"patient": "patient_16_22", "cancer_type": "melanoma", "biopsy_stage": "Screen"},
    "C2D15_16_22": {"patient": "patient_16_22", "cancer_type": "melanoma", "biopsy_stage": "C2D15"},
    "Screen_30_16": {"patient": "patient_30_16", "cancer_type": "melanoma", "biopsy_stage": "Screen"},
    "C2D15_30_16": {"patient": "patient_30_16", "cancer_type": "melanoma", "biopsy_stage": "C2D15"},
    "Screen_23_25": {"patient": "patient_23_25", "cancer_type": "colon_cancer", "biopsy_stage": "Screen"},
    "C2D15_23_25": {"patient": "patient_23_25", "cancer_type": "colon_cancer", "biopsy_stage": "C2D15"},
}

print("pyUCell imported")
print("torch:", str(torch.__version__))
print("CUDA available:", torch.cuda.is_available())


pyUCell imported
torch: 2.11.0+cu130
CUDA available: True


In [3]:
# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
PROJECT_ROOT = Path("/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057")
PIPELINE_ROOT = PROJECT_ROOT / "tmp" / "proseg_resolvi_immune_enrichment_v1"
RESOLVI_ROOT = PIPELINE_ROOT / "02_resolvi"
ALLCELL_ZARR_ROOT = PIPELINE_ROOT / "03a_resolvi_allcell_zarr"
REFERENCE_ROOT = PIPELINE_ROOT / "03b_reference_annotations"
RESCUE_ROOT = PIPELINE_ROOT / "03c_reference_ucell_rescue"
RESCUE_ROOT.mkdir(parents=True, exist_ok=True)

INCLUDE_COLON = True
SECTION_NAMES = [
    sample
    for sample, meta in SAMPLE_INFO.items()
    if INCLUDE_COLON or meta["cancer_type"] != "colon_cancer"
]
# SECTION_NAMES = ["C2D15_23_25"]

CORRECTED_LAYER = "resolvi_corrected_10k"
UCELL_CELL_CHUNK = 1_000
UCELL_MAX_RANK = 1_500
UCELL_MISSING_GENES = "skip"
UCELL_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
UCELL_TIES_METHOD = "min" if UCELL_DEVICE != "cpu" else "average"

# Candidate thresholds are deliberately sensitive and sample-adaptive.
T_SCORE_CANDIDATE_QUANTILE = 0.90
NK_SCORE_CANDIDATE_QUANTILE = 0.90
MIN_T_SCORE = 0.01
MIN_NK_SCORE = 0.01

PANHUMAN_CONFIDENCE_THRESHOLD = 0.50
PLOT_DPI = 500
PLOT_MAX_CELLS = 250_000

USE_EXISTING_SCORED = True
OVERWRITE_SCORED = False
CONTINUE_ON_ERROR = True
PIPELINE_VERSION = "2026-07-29-step3c-reference-ucell-rescue-v1"

print("UCell device:", UCELL_DEVICE)
print("Samples:", SECTION_NAMES)


UCell device: cuda
Samples: ['Screen_39_21', 'C2D15_39_21', 'Screen_17_26', 'C2D15_17_26', 'Screen_18_23', 'C2D15_18_23', 'Screen_16_22', 'C2D15_16_22', 'Screen_30_16', 'C2D15_30_16', 'Screen_23_25', 'C2D15_23_25']


In [4]:
# ---------------------------------------------------------------------
# Signatures
# ---------------------------------------------------------------------
SIGNATURES = {
    "immune_core": [
        "PTPRC", "CD3D", "CD3E", "TRAC", "LCK", "IL32", "LST1",
        "TYROBP", "FCER1G", "MS4A1", "CD79A", "NKG7", "KLRD1",
    ],
    "T_cell_core": [
        "CD3D", "CD3E", "TRAC", "TRBC1", "TRBC2", "CD247", "LCK",
        "LAT", "CD2",
    ],
    "CD4_helper": ["IL7R", "CCR7", "TCF7", "LEF1", "MAL", "LTB", "CD4"],
    "CD8_T": ["CD8A", "CD8B", "CTSW"],
    "Treg": ["FOXP3", "IL2RA", "CTLA4", "TIGIT", "IKZF2"],
    "NK": ["KLRD1", "GNLY", "XCL1", "XCL2", "FCER1G", "TYROBP", "NKG7"],
    "B_cell": ["MS4A1", "CD79A", "CD79B", "CD37", "CD74", "CD22"],
    "plasma_cell": ["MZB1", "JCHAIN", "SDC1", "XBP1", "IGHG1", "IGKC"],
    "myeloid": ["LST1", "TYROBP", "FCER1G", "AIF1", "CTSS", "LYZ"],
    "mast_cell": ["TPSAB1", "TPSB2", "KIT", "MS4A2", "CPA3"],
    "endothelial": [
        "PECAM1", "VWF", "KDR", "ESAM", "ENG", "EMCN", "RAMP2",
        "RGCC", "PLVAP", "CA4",
    ],
    "epithelial_keratin": [
        "EPCAM", "TACSTD2", "KRT7", "KRT8", "KRT18", "KRT19", "MUC1",
        "KRT5", "KRT6A", "KRT6B", "KRT14", "KRT17", "TP63", "SFN",
        "DSG3", "CEACAM5", "CEACAM6", "MSLN",
    ],
    "melanoma_melanocytic": ["MLANA", "PMEL", "TYR", "DCT", "MITF", "SOX10", "S100B"],
    "melanoma_dedifferentiated": ["AXL", "NGFR", "SOX9"],
    "fibroblast": ["COL1A1", "COL1A2", "COL3A1", "DCN", "LUM", "PDGFRA", "C7"],
    "mural": ["RGS5", "CSPG4", "MCAM", "PDGFRB", "ACTA2", "TAGLN", "MYL9"],
    "erythroid": ["HBB", "HBA1", "HBA2", "ALAS2", "GYPA", "AHSP"],
}

T_SPECIFIC_GENES = ["CD3D", "CD3E", "TRAC", "TRBC1", "TRBC2", "CD247"]
T_SUPPORTING_GENES = ["CD2", "CD7", "LCK", "LAT", "IL32", "MAL", "LTB"]

TARGET_SIGNATURES = ["immune_core", "endothelial"]
EXCLUSION_SIGNATURES = [
    "epithelial_keratin",
    "melanoma_melanocytic",
    "melanoma_dedifferentiated",
    "fibroblast",
    "mural",
    "erythroid",
]


In [5]:
# ---------------------------------------------------------------------
# Paths and helpers
# ---------------------------------------------------------------------
def paths_for_sample(sample: str) -> dict[str, Path]:
    out = RESCUE_ROOT / sample
    out.mkdir(parents=True, exist_ok=True)
    return {
        "raw_h5ad": RESOLVI_ROOT / sample / f"{sample}_resolvi_annotated.h5ad",
        "zarr": ALLCELL_ZARR_ROOT / sample / f"{sample}_resolvi_allcells.zarr",
        "zarr_summary": ALLCELL_ZARR_ROOT / sample / f"{sample}_resolvi_allcells_zarr_summary.json",
        "reference": REFERENCE_ROOT / sample / f"{sample}_reference_annotations.parquet",
        "embedding": REFERENCE_ROOT / sample / f"{sample}_panhuman_embedding.npy",
        "out": out,
        "scored": out / f"{sample}_reference_ucell_scored.h5ad",
        "summary": out / f"{sample}_reference_ucell_rescue_summary.json",
        "evidence_counts": out / f"{sample}_T_evidence_counts.csv",
        "evidence_intersections": out / f"{sample}_T_evidence_intersections.csv",
        "tnk_candidates": out / f"{sample}_T_NK_candidate_metadata.parquet",
        "target_candidates": out / f"{sample}_immune_endothelial_candidate_metadata.parquet",
    }


def write_json(payload, path: Path) -> None:
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    temp.replace(path)


def atomic_write_h5ad(adata: ad.AnnData, path: Path) -> None:
    temp = path.with_name(path.stem + ".tmp.h5ad")
    if temp.exists():
        temp.unlink()
    adata.write_h5ad(temp, compression="lzf")
    temp.replace(path)


def names_hash(names) -> str:
    digest = hashlib.sha256()
    for value in names.astype(str):
        digest.update(value.encode("utf-8"))
        digest.update(b"\0")
    return digest.hexdigest()


def map_genes(var_names: pd.Index, genes: list[str]) -> list[str]:
    lookup = {}
    for name in var_names.astype(str):
        lookup.setdefault(name.upper(), name)
    mapped = []
    for gene in genes:
        value = lookup.get(str(gene).upper())
        if value is not None and value not in mapped:
            mapped.append(value)
    return mapped


def raw_detected_count(adata: ad.AnnData, genes: list[str]) -> np.ndarray:
    mapped = map_genes(adata.var_names, genes)
    if not mapped:
        return np.zeros(adata.n_obs, dtype=np.int16)
    matrix = sp.csr_matrix(adata[:, mapped].X)
    return np.asarray((matrix > 0).sum(axis=1)).ravel().astype(np.int16)


def open_corrected_layer(zarr_path: Path):
    root = zarr.open_group(str(zarr_path), mode="r")
    return root["layers"][CORRECTED_LAYER]


def validate_inputs(sample: str, raw: ad.AnnData, paths: dict[str, Path]):
    if not paths["zarr"].exists():
        raise FileNotFoundError(paths["zarr"])
    if not paths["reference"].exists():
        raise FileNotFoundError(paths["reference"])
    summary = json.loads(paths["zarr_summary"].read_text(encoding="utf-8"))
    if summary.get("obs_names_sha256") != names_hash(raw.obs_names):
        raise ValueError("Raw H5AD and corrected Zarr obs_names do not align.")
    if summary.get("var_names_sha256") != names_hash(raw.var_names):
        raise ValueError("Raw H5AD and corrected Zarr var_names do not align.")
    corrected = open_corrected_layer(paths["zarr"])
    if tuple(corrected.shape) != tuple(raw.shape):
        raise ValueError("Corrected layer and raw H5AD have different shapes.")
    return corrected, summary


def map_signatures(var_names: pd.Index):
    mapped = {}
    rows = []
    for name, genes in SIGNATURES.items():
        present = map_genes(var_names, genes)
        mapped[name] = present
        rows.append(
            {
                "signature": name,
                "n_requested": len(genes),
                "n_present": len(present),
                "fraction_present": len(present) / max(len(genes), 1),
                "present_genes": ";".join(present),
            }
        )
    return mapped, pd.DataFrame(rows)


In [6]:
# ---------------------------------------------------------------------
# Corrected-expression UCell scoring in chunks
# ---------------------------------------------------------------------
def corrected_ucell_scores(
    corrected_array,
    obs_names: pd.Index,
    var_names: pd.Index,
    mapped_signatures: dict,
) -> pd.DataFrame:
    signature_names = list(mapped_signatures)
    output = np.zeros((len(obs_names), len(signature_names)), dtype=np.float32)

    for start in range(0, len(obs_names), int(UCELL_CELL_CHUNK)):
        end = min(start + int(UCELL_CELL_CHUNK), len(obs_names))
        expression = np.asarray(corrected_array[start:end, :], dtype=np.float32)
        if not np.isfinite(expression).all():
            raise FloatingPointError("Corrected Zarr chunk contains non-finite values.")

        temp = ad.AnnData(
            X=expression,
            obs=pd.DataFrame(index=obs_names[start:end]),
            var=pd.DataFrame(index=var_names.copy()),
        )
        uc.compute_ucell_scores(
            temp,
            signatures=mapped_signatures,
            layer=None,
            max_rank=min(int(UCELL_MAX_RANK), temp.n_vars),
            ties_method=UCELL_TIES_METHOD,
            missing_genes=UCELL_MISSING_GENES,
            chunk_size=None,
            suffix="_resolvi_UCell",
            n_jobs=1,
            device=UCELL_DEVICE,
        )
        for column_index, signature in enumerate(signature_names):
            output[start:end, column_index] = temp.obs[
                f"{signature}_resolvi_UCell"
            ].to_numpy(dtype=np.float32)

        print(f"UCell scored {end:,}/{len(obs_names):,} cells")
        del expression, temp
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(
        output,
        index=obs_names,
        columns=[f"{name}_resolvi_UCell" for name in signature_names],
    )


In [7]:
# ---------------------------------------------------------------------
# Candidate evidence and diagnostic plots
# ---------------------------------------------------------------------
def add_composite_and_rescue_columns(adata: ad.AnnData) -> dict:
    t_score = adata.obs["T_cell_core_resolvi_UCell"].to_numpy(dtype=float)
    nk_score = adata.obs["NK_resolvi_UCell"].to_numpy(dtype=float)
    t_threshold = max(float(MIN_T_SCORE), float(np.quantile(t_score, T_SCORE_CANDIDATE_QUANTILE)))
    nk_threshold = max(float(MIN_NK_SCORE), float(np.quantile(nk_score, NK_SCORE_CANDIDATE_QUANTILE)))

    adata.obs["corrected_t_score_candidate"] = t_score >= t_threshold
    adata.obs["corrected_nk_score_candidate"] = nk_score >= nk_threshold
    adata.obs["T_vs_NK_margin"] = t_score - nk_score
    adata.obs["T_vs_myeloid_margin"] = (
        t_score - adata.obs["myeloid_resolvi_UCell"].to_numpy(dtype=float)
    )

    adata.obs["raw_t_specific_detected"] = raw_detected_count(
        adata, T_SPECIFIC_GENES
    )
    adata.obs["raw_t_supporting_detected"] = raw_detected_count(
        adata, T_SUPPORTING_GENES
    )
    adata.obs["raw_t_coherent"] = (
        (adata.obs["raw_t_specific_detected"].to_numpy(dtype=int) >= 2)
        | (
            (adata.obs["raw_t_specific_detected"].to_numpy(dtype=int) >= 1)
            & (adata.obs["raw_t_supporting_detected"].to_numpy(dtype=int) >= 1)
        )
    )

    evidence_columns = [
        "raw_t_coherent",
        "corrected_t_score_candidate",
        "panhuman_t_candidate",
        "celltypist_high_t_candidate",
        "celltypist_low_t_candidate",
    ]
    evidence = np.column_stack(
        [adata.obs[column].fillna(False).to_numpy(dtype=bool) for column in evidence_columns]
    )
    adata.obs["t_evidence_count"] = evidence.sum(axis=1).astype(np.int8)
    adata.obs["t_candidate_union"] = adata.obs["t_evidence_count"] >= 1
    adata.obs["t_candidate_multi_evidence"] = adata.obs["t_evidence_count"] >= 2

    high_confidence = (
        adata.obs["raw_t_coherent"].to_numpy(dtype=bool)
        & (
            adata.obs["corrected_t_score_candidate"].to_numpy(dtype=bool)
            | adata.obs["panhuman_t_candidate"].to_numpy(dtype=bool)
            | adata.obs["celltypist_high_t_candidate"].to_numpy(dtype=bool)
            | adata.obs["celltypist_low_t_candidate"].to_numpy(dtype=bool)
        )
    ) | (
        adata.obs["panhuman_t_high_confidence"].to_numpy(dtype=bool)
        & (
            adata.obs["celltypist_high_t_candidate"].to_numpy(dtype=bool)
            | adata.obs["celltypist_low_t_candidate"].to_numpy(dtype=bool)
        )
        & (adata.obs["raw_t_specific_detected"].to_numpy(dtype=int) >= 1)
    )
    adata.obs["t_candidate_high_confidence"] = high_confidence

    adata.obs["tnk_candidate_union"] = (
        adata.obs["t_candidate_union"].to_numpy(dtype=bool)
        | adata.obs["corrected_nk_score_candidate"].to_numpy(dtype=bool)
        | adata.obs["panhuman_nk_candidate"].to_numpy(dtype=bool)
        | adata.obs["celltypist_high_nk_candidate"].to_numpy(dtype=bool)
        | adata.obs["celltypist_low_nk_candidate"].to_numpy(dtype=bool)
    )

    target_score = np.maximum(
        adata.obs["immune_core_resolvi_UCell"].to_numpy(dtype=float),
        adata.obs["endothelial_resolvi_UCell"].to_numpy(dtype=float),
    )
    exclusion_matrix = np.column_stack(
        [
            adata.obs[f"{signature}_resolvi_UCell"].to_numpy(dtype=float)
            for signature in EXCLUSION_SIGNATURES
        ]
    )
    adata.obs["target_score"] = target_score.astype(np.float32)
    adata.obs["exclusion_score"] = exclusion_matrix.max(axis=1).astype(np.float32)
    adata.obs["target_margin"] = (
        adata.obs["target_score"].to_numpy(dtype=float)
        - adata.obs["exclusion_score"].to_numpy(dtype=float)
    ).astype(np.float32)

    adata.obs["immune_endothelial_candidate_union"] = (
        adata.obs["t_candidate_union"].to_numpy(dtype=bool)
        | (
            adata.obs["immune_core_resolvi_UCell"].to_numpy(dtype=float)
            >= np.quantile(
                adata.obs["immune_core_resolvi_UCell"].to_numpy(dtype=float), 0.90
            )
        )
        | (
            adata.obs["endothelial_resolvi_UCell"].to_numpy(dtype=float)
            >= np.quantile(
                adata.obs["endothelial_resolvi_UCell"].to_numpy(dtype=float), 0.90
            )
        )
    )

    return {
        "t_score_candidate_threshold": t_threshold,
        "nk_score_candidate_threshold": nk_threshold,
        "t_score_candidate_quantile": float(T_SCORE_CANDIDATE_QUANTILE),
        "nk_score_candidate_quantile": float(NK_SCORE_CANDIDATE_QUANTILE),
        "evidence_columns": evidence_columns,
    }


def evidence_count_table(adata: ad.AnnData) -> pd.DataFrame:
    columns = [
        "raw_t_coherent",
        "corrected_t_score_candidate",
        "panhuman_t_candidate",
        "panhuman_t_high_confidence",
        "celltypist_high_t_candidate",
        "celltypist_low_t_candidate",
        "t_candidate_union",
        "t_candidate_multi_evidence",
        "t_candidate_high_confidence",
        "tnk_candidate_union",
    ]
    return pd.DataFrame(
        {
            "evidence": columns,
            "n_cells": [int(adata.obs[column].fillna(False).sum()) for column in columns],
            "fraction": [float(adata.obs[column].fillna(False).mean()) for column in columns],
        }
    )


def evidence_intersection_table(adata: ad.AnnData) -> pd.DataFrame:
    pairs = [
        ("raw_t_coherent", "panhuman_t_candidate"),
        ("raw_t_coherent", "corrected_t_score_candidate"),
        ("panhuman_t_candidate", "celltypist_high_t_candidate"),
        ("panhuman_t_candidate", "celltypist_low_t_candidate"),
        ("corrected_t_score_candidate", "celltypist_low_t_candidate"),
    ]
    rows = []
    for left, right in pairs:
        left_values = adata.obs[left].fillna(False).to_numpy(dtype=bool)
        right_values = adata.obs[right].fillna(False).to_numpy(dtype=bool)
        rows.append(
            {
                "left": left,
                "right": right,
                "intersection": int((left_values & right_values).sum()),
                "union": int((left_values | right_values).sum()),
            }
        )
    return pd.DataFrame(rows)


def spatial_coordinates(adata: ad.AnnData):
    for key in ("X_spatial", "spatial", "spatial_fullres"):
        if key in adata.obsm:
            coords = np.asarray(adata.obsm[key], dtype=float)
            if coords.shape == (adata.n_obs, 2):
                return coords, key
    return None, None


def save_plots(adata: ad.AnnData, sample: str, out: Path) -> None:
    out.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(7, 6))
    hb = ax.hexbin(
        adata.obs["T_cell_core_resolvi_UCell"].to_numpy(dtype=float),
        adata.obs["NK_resolvi_UCell"].to_numpy(dtype=float),
        gridsize=100,
        mincnt=1,
    )
    ax.set_xlabel("T-cell core ResolVI UCell")
    ax.set_ylabel("NK ResolVI UCell")
    ax.set_title(f"{sample}: corrected T versus NK evidence")
    fig.colorbar(hb, ax=ax, label="Cells")
    fig.tight_layout()
    fig.savefig(out / f"{sample}_T_vs_NK_UCell.png", dpi=PLOT_DPI, bbox_inches="tight")
    plt.close(fig)

    counts = adata.obs["t_evidence_count"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(7, 5))
    counts.plot(kind="bar", ax=ax)
    ax.set_xlabel("Number of independent T-cell evidence streams")
    ax.set_ylabel("Cells")
    ax.set_title(f"{sample}: T-cell rescue evidence")
    ax.tick_params(axis="x", rotation=0)
    fig.tight_layout()
    fig.savefig(out / f"{sample}_T_evidence_count.png", dpi=PLOT_DPI, bbox_inches="tight")
    plt.close(fig)

    coords, coord_key = spatial_coordinates(adata)
    if coords is None:
        return
    rng = np.random.default_rng(0)
    if adata.n_obs > PLOT_MAX_CELLS:
        indices = np.sort(rng.choice(adata.n_obs, PLOT_MAX_CELLS, replace=False))
    else:
        indices = np.arange(adata.n_obs)

    for column in (
        "T_cell_core_resolvi_UCell",
        "t_evidence_count",
        "t_candidate_multi_evidence",
        "tnk_candidate_union",
    ):
        values = adata.obs[column].iloc[indices]
        fig, ax = plt.subplots(figsize=(8, 8))
        scatter = ax.scatter(
            coords[indices, 0],
            coords[indices, 1],
            c=values.astype(float),
            s=1,
            linewidths=0,
        )
        ax.set_aspect("equal")
        ax.invert_yaxis()
        ax.set_title(f"{sample}: {column} ({coord_key})")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        fig.colorbar(scatter, ax=ax, label=column)
        fig.tight_layout()
        fig.savefig(out / f"{sample}_spatial_{column}.png", dpi=PLOT_DPI, bbox_inches="tight")
        plt.close(fig)


In [8]:
# ---------------------------------------------------------------------
# Per-sample rescue runner
# ---------------------------------------------------------------------
def process_rescue_sample(sample: str) -> dict:
    paths = paths_for_sample(sample)
    print("\n" + "=" * 90)
    print("Reference/UCell rescue sample:", sample)

    for key in ("raw_h5ad", "zarr", "zarr_summary", "reference"):
        if not paths[key].exists():
            raise FileNotFoundError(paths[key])

    if (
        USE_EXISTING_SCORED
        and not OVERWRITE_SCORED
        and paths["scored"].exists()
        and paths["summary"].exists()
    ):
        summary = json.loads(paths["summary"].read_text(encoding="utf-8"))
        if summary.get("pipeline_version") == PIPELINE_VERSION:
            print("Reusing existing scored rescue object.")
            return summary

    started = time.time()
    adata = ad.read_h5ad(paths["raw_h5ad"])
    corrected, zarr_summary = validate_inputs(sample, adata, paths)

    reference = pd.read_parquet(paths["reference"])
    reference.index = reference.index.astype(str)
    missing = adata.obs_names.difference(reference.index)
    if len(missing):
        raise KeyError(f"{len(missing):,} cells missing from reference annotations.")
    reference = reference.reindex(adata.obs_names)
    duplicate_columns = adata.obs.columns.intersection(reference.columns)
    if len(duplicate_columns):
        adata.obs = adata.obs.drop(columns=list(duplicate_columns))
    adata.obs = pd.concat([adata.obs, reference], axis=1)

    if paths["embedding"].exists():
        embedding = np.load(paths["embedding"], mmap_mode="r")
        if embedding.shape[0] != adata.n_obs:
            raise ValueError("Pan-Human embedding row count does not match cells.")
        adata.obsm["X_panhuman"] = np.asarray(embedding, dtype=np.float32)
        del embedding

    mapped_signatures, coverage = map_signatures(adata.var_names)
    coverage.to_csv(paths["out"] / f"{sample}_signature_coverage.csv", index=False)
    scores = corrected_ucell_scores(
        corrected,
        adata.obs_names,
        adata.var_names,
        mapped_signatures,
    )
    adata.obs = pd.concat([adata.obs, scores], axis=1)

    thresholds = add_composite_and_rescue_columns(adata)
    evidence_counts = evidence_count_table(adata)
    intersections = evidence_intersection_table(adata)
    evidence_counts.to_csv(paths["evidence_counts"], index=False)
    intersections.to_csv(paths["evidence_intersections"], index=False)

    adata.uns["reference_ucell_rescue"] = {
        "pipeline_version": PIPELINE_VERSION,
        "sample": sample,
        **SAMPLE_INFO[sample],
        "corrected_expression_source": str(paths["zarr"]),
        "corrected_layer": CORRECTED_LAYER,
        "raw_count_location": "X",
        "signatures": SIGNATURES,
        "candidate_thresholds": thresholds,
        "interpretation": (
            "Candidate labels are sensitivity-oriented evidence unions; "
            "they are not final cell-type annotations."
        ),
    }

    save_plots(adata, sample, paths["out"])
    atomic_write_h5ad(adata, paths["scored"])

    tnk_columns = [
        column
        for column in adata.obs.columns
        if (
            column.startswith("panhuman_")
            or column.startswith("celltypist_")
            or column.endswith("_resolvi_UCell")
            or column.startswith("raw_t_")
            or column.startswith("t_")
            or column.startswith("tnk_")
            or column in ("T_vs_NK_margin", "T_vs_myeloid_margin")
        )
    ]
    tnk = adata.obs.loc[
        adata.obs["tnk_candidate_union"].to_numpy(dtype=bool),
        tnk_columns,
    ].copy()
    tnk.to_parquet(paths["tnk_candidates"])

    target = adata.obs.loc[
        adata.obs["immune_endothelial_candidate_union"].to_numpy(dtype=bool)
    ].copy()
    target.to_parquet(paths["target_candidates"])

    summary = {
        "pipeline_version": PIPELINE_VERSION,
        "sample": sample,
        **SAMPLE_INFO[sample],
        "n_cells": int(adata.n_obs),
        "n_genes": int(adata.n_vars),
        "scored_h5ad": str(paths["scored"]),
        "corrected_zarr": str(paths["zarr"]),
        "n_raw_t_coherent": int(adata.obs["raw_t_coherent"].sum()),
        "n_corrected_t_score_candidate": int(
            adata.obs["corrected_t_score_candidate"].sum()
        ),
        "n_panhuman_t_candidate": int(adata.obs["panhuman_t_candidate"].sum()),
        "n_t_candidate_union": int(adata.obs["t_candidate_union"].sum()),
        "n_t_candidate_multi_evidence": int(
            adata.obs["t_candidate_multi_evidence"].sum()
        ),
        "n_t_candidate_high_confidence": int(
            adata.obs["t_candidate_high_confidence"].sum()
        ),
        "n_tnk_candidate_union": int(adata.obs["tnk_candidate_union"].sum()),
        "n_immune_endothelial_candidate_union": int(
            adata.obs["immune_endothelial_candidate_union"].sum()
        ),
        "thresholds": thresholds,
        "runtime_minutes": (time.time() - started) / 60.0,
    }
    write_json(summary, paths["summary"])
    print("Saved:", paths["scored"])

    del adata, reference, scores, coverage, evidence_counts, intersections, corrected
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary


## Run selected samples

The output is an exploratory rescue layer. Review candidate counts, spatial
plots, and intersections before the next step: cancer-specific T/NK integration
and unsupervised subclustering on corrected expression.


In [9]:
rescue_results = {}
rescue_failures = {}

for sample in SECTION_NAMES:
    try:
        rescue_results[sample] = process_rescue_sample(sample)
    except Exception as exc:
        rescue_failures[sample] = repr(exc)
        print(f"[FAILED] {sample}: {type(exc).__name__}: {exc}")
        if not CONTINUE_ON_ERROR:
            raise
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

pd.DataFrame.from_dict(rescue_results, orient="index").to_csv(
    RESCUE_ROOT / "all_samples_reference_ucell_rescue_summary.csv"
)
write_json(
    rescue_failures,
    RESCUE_ROOT / "all_samples_reference_ucell_rescue_failures.json",
)
print("Completed:", sorted(rescue_results))
print("Failures:", json.dumps(rescue_failures, indent=2))



Reference/UCell rescue sample: Screen_39_21
[FAILED] Screen_39_21: TypeError: compute_ucell_scores() got an unexpected keyword argument 'device'

Reference/UCell rescue sample: C2D15_39_21
[FAILED] C2D15_39_21: TypeError: compute_ucell_scores() got an unexpected keyword argument 'device'

Reference/UCell rescue sample: Screen_17_26
[FAILED] Screen_17_26: TypeError: compute_ucell_scores() got an unexpected keyword argument 'device'

Reference/UCell rescue sample: C2D15_17_26
[FAILED] C2D15_17_26: TypeError: compute_ucell_scores() got an unexpected keyword argument 'device'

Reference/UCell rescue sample: Screen_18_23
[FAILED] Screen_18_23: TypeError: compute_ucell_scores() got an unexpected keyword argument 'device'

Reference/UCell rescue sample: C2D15_18_23
[FAILED] C2D15_18_23: TypeError: compute_ucell_scores() got an unexpected keyword argument 'device'

Reference/UCell rescue sample: Screen_16_22
[FAILED] Screen_16_22: TypeError: compute_ucell_scores() got an unexpected keyword ar

In [10]:
%load_ext watermark

In [11]:
#print environment and packages used
%watermark -d -i -m -iv -v --gpu

Date: 2026-07-29

Python implementation: CPython
Python version       : 3.13.7
IPython version      : 9.12.0

Compiler    : Clang 20.1.4 
OS          : Linux
Release     : 6.1.158-178.288.amzn2023.x86_64
Machine     : x86_64
Processor   : x86_64
CPU cores   : 96
Architecture: 64bit

anndata   : 0.12.11
json      : 2.0.9
matplotlib: 3.10.8
numpy     : 2.4.4
pandas    : 2.3.3
pyucell   : 0.5.0
scipy     : 1.17.1
torch     : 2.11.0
zarr      : 3.1.6

GPU Info: 
  GPU 0: NVIDIA A10G
  GPU 1: NVIDIA A10G
  GPU 2: NVIDIA A10G
  GPU 3: NVIDIA A10G

